# Olist Marketplace – Silver Cleaning

## Customers, Orders & Order Reviews

This notebook combines the three existing preprocessing notebooks for the Silver Layer:

1. **Customers**
2. **Orders**
3. **Order Reviews**

The three sections remain clearly separated both conceptually and within the notebook.

The setup, path definitions, and loading of the Bronze datasets are performed **only once**.

The existing cleaning and validation logic of the three individual notebooks is retained. At the end, all three validated Silver datasets are exported together to the `processed` folder.

## Common Setup & Bronze Data Load

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_PATH = Path("../datasets/raw")
PROCESSED_PATH = Path("../datasets/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)


In [2]:
# Load all Bronze datasets once

bronze_customers = pd.read_csv(
    RAW_PATH / "olist_customers_dataset.csv"
)

bronze_orders = pd.read_csv(
    RAW_PATH / "olist_orders_dataset.csv"
)

bronze_order_reviews = pd.read_csv(
    RAW_PATH / "olist_order_reviews_dataset_en.csv"
)

# Preserve raw row counts for later validation
customers_raw_row_count = len(bronze_customers)
orders_raw_row_count = len(bronze_orders)
order_reviews_raw_row_count = len(bronze_order_reviews)

print(f"Bronze customers: {bronze_customers.shape}")
print(f"Bronze orders: {bronze_orders.shape}")
print(f"Bronze order reviews: {bronze_order_reviews.shape}")


Bronze customers: (99441, 5)
Bronze orders: (99441, 8)
Bronze order reviews: (99224, 9)



# S3 SETUP & BRONZE DATA LOAD – FUTURE LAMBDA VERSION

This block is intentionally commented out.
It will be activated when the preprocessing pipeline is moved from local processing to AWS Lambda.


In [ ]:

# Data flow:
#
# S3 Bucket
#     └── raw/
#          ├── olist_customers_dataset.csv
#          ├── olist_orders_dataset.csv
#          └── olist_order_reviews_dataset.csv
#
#                 ↓
#
#              AWS Lambda
#
#                 ↓
#
# S3 Bucket
#     └── processed/
#          ├── silver_customers.csv
#          ├── silver_orders.csv
#          └── silver_order_reviews_en.csv
#
# ============================================================


# import boto3
# from io import StringIO


# ------------------------------------------------------------
# S3 Configuration
# ------------------------------------------------------------

# S3_BUCKET = "dsi-final-project-360964564955-eu-central-1-an"

# RAW_PREFIX = "raw/"
# PROCESSED_PREFIX = "processed/"


# ------------------------------------------------------------
# S3 Client
# ------------------------------------------------------------

# s3 = boto3.client("s3")


# ------------------------------------------------------------
# Load Bronze datasets from S3
# ------------------------------------------------------------

# def load_csv_from_s3(bucket, key):
#     response = s3.get_object(
#         Bucket=bucket,
#         Key=key
#     )
#
#     return pd.read_csv(response["Body"])


# bronze_customers = load_csv_from_s3(
#     S3_BUCKET,
#     f"{RAW_PREFIX}olist_customers_dataset.csv"
# )
#
# bronze_orders = load_csv_from_s3(
#     S3_BUCKET,
#     f"{RAW_PREFIX}olist_orders_dataset.csv"
# )
#
# bronze_order_reviews = load_csv_from_s3(
#     S3_BUCKET,
#     f"{RAW_PREFIX}olist_order_reviews_dataset_en.csv"
# )


# ------------------------------------------------------------
# Preserve raw row counts for later validation
# ------------------------------------------------------------

# customers_raw_row_count = len(bronze_customers)
# orders_raw_row_count = len(bronze_orders)
# order_reviews_raw_row_count = len(bronze_order_reviews)


# print(f"Bronze customers: {bronze_customers.shape}")
# print(f"Bronze orders: {bronze_orders.shape}")
# print(f"Bronze order reviews: {bronze_order_reviews.shape}")

# CUSTOMERS – Silver Preprocessing

---

**Dataset:** `olist_customers_dataset.csv`  
**Silver output:** `silver_customers.csv`

This section is structured independently from the other functional areas and uses the shared setup.

## 1. Data Audit

The raw dataset is inspected before any transformation is applied.

The audit covers:

- dataset dimensions
- column names
- data types
- missing values
- duplicate records
- key uniqueness
- customer identifier cardinality
- categorical values
- ZIP code plausibility

In [3]:
#Check shape
bronze_customers.shape

(99441, 5)

In [4]:
#Check columns
bronze_customers.columns.tolist()

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [5]:
#Check datatypes
bronze_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB


In [6]:
#Check missing values
bronze_customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [7]:
bronze_customers.isna().mean().mul(100).round(2)

customer_id                 0.0
customer_unique_id          0.0
customer_zip_code_prefix    0.0
customer_city               0.0
customer_state              0.0
dtype: float64

In [8]:
#Check duplicates
bronze_customers.duplicated().sum()

np.int64(0)

In [9]:
bronze_customers["customer_id"].nunique()

99441

In [10]:
len(bronze_customers)

99441

In [11]:
#Check primary keys
bronze_customers["customer_id"].is_unique

True

In [12]:
#Check unique ids
bronze_customers["customer_unique_id"].nunique()

96096

In [13]:
customer_counts = (
    bronze_customers
    .groupby("customer_unique_id")
    .size()
    .sort_values(ascending=False))

customer_counts.head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
63cfc61cee11cbe306bff5857d00bfe4     6
dc813062e0fc23409cd255f7f53c7074     6
de34b16117594161a6a89c50b289d35a     6
dtype: int64

In [14]:
#How many unique customers have multiple customer records?
(customer_counts > 1).sum()

np.int64(2997)

In [15]:
#How many customers have more than one record?
customer_counts.value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [16]:
repeat_customer_records = customer_counts[customer_counts > 1]

repeat_customer_records.describe()

count    2997.000000
mean        2.116116
std         0.516610
min         2.000000
25%         2.000000
50%         2.000000
75%         2.000000
max        17.000000
dtype: float64

In [17]:
repeat_customer_records.sum()

np.int64(6342)

### Customer Record Cardinality

The dataset contains 96,096 unique `customer_unique_id` values across
99,441 customer records.

2,997 customers have more than one customer record, accounting for
6,342 customer records in total.

These repeated `customer_unique_id` values are not considered duplicate
records because `customer_id` remains unique and represents the
customer-record level of this table.

In [18]:
#Check customer state
bronze_customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

In [19]:
bronze_customers["customer_city"].nunique()

4119

In [20]:
#Check zip-code
bronze_customers["customer_zip_code_prefix"].describe()

count    99441.000000
mean     35137.474583
std      29797.938996
min       1003.000000
25%      11347.000000
50%      24416.000000
75%      58900.000000
max      99990.000000
Name: customer_zip_code_prefix, dtype: float64

In [21]:
bronze_customers["customer_zip_code_prefix"].nunique()

14994

In [22]:
(bronze_customers["customer_zip_code_prefix"] < 0).sum()

np.int64(0)

In [23]:
bronze_customers["customer_zip_code_prefix"].isna().sum()

np.int64(0)

### Audit Findings

The raw customers dataset contains 99,441 records and 5 columns.

No missing values were identified and no fully duplicated rows were found.

`customer_id` is unique across all records and therefore represents the
primary key at the customer-record level.

`customer_unique_id` contains repeated values. This is expected because
multiple customer records can belong to the same underlying customer.
Therefore, `customer_unique_id` must not be treated as the primary key of
this table.

The dataset contains 27 state codes, 4,119 unique cities and 14,994 ZIP
code prefixes.

No structural data quality issue requiring record removal was identified

## 2. Cleaning & Standardization

The raw dataset is preserved and all transformations are applied to a
separate working copy.

Because the audit identified no missing values, duplicate records or
primary-key violations, no customer records are removed during
preprocessing.

The following transformations are applied:

- remove leading and trailing whitespace from string fields
- standardize state codes to uppercase
- explicitly define data types
- preserve the original city naming rather than applying aggressive
  text normalization

In [24]:
#Create copy
silver_customers = bronze_customers.copy()

In [25]:
# Create Silver Unique Key
silver_customers.insert(
    0,
    "id_customers",
    range(1, len(silver_customers) + 1)
)


In [26]:
#Check columns
silver_customers.columns.tolist()

['id_customers',
 'customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

### Silver Unique Key

A technical unique key `id_customers` is created for the Silver table.

The original source identifiers `customer_id` and `customer_unique_id` are retained unchanged.

`id_customers` is generated as a sequential integer starting at 1 and serves as the technical unique identifier of each Silver record.

In [27]:
#Identifying string columns and removing spaces
string_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"]

for col in string_columns:
    silver_customers[col] = silver_customers[col].str.strip()

In [28]:
#Normalize 'state' and check
silver_customers["customer_state"] = (
   silver_customers["customer_state"].str.lower())

In [29]:
silver_customers["customer_state"].unique()

<ArrowStringArray>
['sp', 'sc', 'mg', 'pr', 'rj', 'rs', 'pa', 'go', 'es', 'ba', 'ma', 'ms', 'ce',
 'df', 'rn', 'pe', 'mt', 'am', 'ap', 'al', 'ro', 'pb', 'to', 'pi', 'ac', 'se',
 'rr']
Length: 27, dtype: str

In [30]:
silver_customers["customer_state"].nunique()

27

In [31]:
#Normalize 'city'
silver_customers["customer_city"] = (
    silver_customers["customer_city"].str.strip())

### City Standardization Decision

City values are only stripped of leading and trailing whitespace.

No lower-case, title-case or character replacement is applied because
aggressive normalization could alter geographic names or remove
meaningful linguistic information.

Further geographic standardization, if required, should be handled as a
separate data-quality or geospatial transformation.

In [32]:
#Define datatypes
#IDs
silver_customers["customer_id"] = (
    silver_customers["customer_id"].astype("string"))

silver_customers["customer_unique_id"] = (
    silver_customers["customer_unique_id"].astype("string"))

In [33]:
#City
silver_customers["customer_city"] = (
    silver_customers["customer_city"].astype("string"))

In [34]:
#State
silver_customers["customer_state"] = (
    silver_customers["customer_state"].astype("string"))

In [35]:
#ZIP Prefix
silver_customers["customer_zip_code_prefix"] = (
    silver_customers["customer_zip_code_prefix"]
    .astype("string")
    .str.zfill(5))

In [36]:
#Check datatypes
silver_customers.dtypes

id_customers                 int64
customer_id                 string
customer_unique_id          string
customer_zip_code_prefix    string
customer_city               string
customer_state              string
dtype: object

In [37]:
#Compare raw and cleaned dataset
print("Raw rows:", len(silver_customers))
print("Silver rows:", len(silver_customers))

Raw rows: 99441
Silver rows: 99441


In [38]:
assert len(silver_customers) == len(silver_customers)

In [39]:
#Check if new missing values appeared
silver_customers.isna().sum()

id_customers                0
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [40]:
# Standardize city names:
# - convert to lowercase
# - replace hyphens with spaces
# - remove apostrophes
# - remove leading/trailing whitespace
# - normalize multiple spaces

silver_customers["customer_city"] = (
    silver_customers["customer_city"]
    .str.lower()
    .str.replace("-", " ", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [41]:
# Replace incorrect city value
silver_customers.loc[
    silver_customers["customer_city"] == "quilometro 14 do mutum",
    "customer_city"
] = "mutum"

In [42]:
print(
    silver_customers.loc[
        silver_customers["customer_city"] == "mutum",
        "customer_city"
    ].value_counts()
)

customer_city
mutum    9
Name: count, dtype: int64[pyarrow]


In [43]:
#State quality check
valid_states = {
    'sp', 'sc', 'mg', 'pr', 'rj', 'rs', 'pa', 'go', 'es', 'ba', 'ma', 'ms', 'ce',
    'df', 'rn', 'pe', 'mt', 'am', 'ap', 'al', 'ro', 'pb', 'to', 'pi', 'ac', 'se',
    'rr'
}

invalid_states = set(silver_customers["customer_state"].dropna().unique()) - valid_states

invalid_states

set()

In [44]:
#ZIP Prefix quality check

silver_customers["customer_zip_code_prefix"].str.len().eq(5).all()
silver_customers["customer_zip_code_prefix"].str.isnumeric().all()


np.True_

In [45]:
#Primary-Key quality check
assert silver_customers["customer_id"].notna().all()

assert silver_customers["customer_id"].is_unique

## 3. Silver Data Quality Validation Customers

Before exporting the Silver dataset, a quality gate is applied.

The Silver dataset must satisfy the following requirements:

- record count remains unchanged
- `customer_id` is non-null
- `customer_id` is unique
- `customer_unique_id` is non-null
- `customer_state` is non-null
- no missing values remain
- all state codes are valid
- ZIP code prefixes fall within the defined plausibility range

In [46]:
expected_columns = [
    "id_customers",
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

assert silver_customers.columns.tolist() == expected_columns

assert len(silver_customers) == len(bronze_customers)

assert silver_customers["id_customers"].notna().all()
assert silver_customers["id_customers"].is_unique
assert silver_customers["id_customers"].min() == 1
assert silver_customers["id_customers"].max() == len(silver_customers)

assert silver_customers["customer_id"].notna().all()
assert silver_customers["customer_id"].is_unique

assert silver_customers["customer_unique_id"].notna().all()

assert silver_customers["customer_city"].str.match(r"^[a-z\s]+$").all()

assert (silver_customers["customer_city"]== silver_customers["customer_city"].str.lower()).all()

assert silver_customers["customer_state"].notna().all()

assert silver_customers.isna().sum().sum() == 0

assert invalid_states == set()

assert silver_customers["customer_zip_code_prefix"].str.len().eq(5).all()
assert silver_customers["customer_zip_code_prefix"].str.isnumeric().all()

print("Silver data quality checks passed successfully.")

Silver data quality checks passed successfully.


## 4. Customers - Preprocessing Summary

The customers dataset successfully passed the preprocessing quality gate.

### Result

- 99,441 customer records retained
- no records removed
- no missing values identified
- `id_customers` integrated and validated as unique key
- `customer_unique_id` correctly treated as the underlying customer identifier
- `customer_id` validated as the primary key at customer-record level
- state codes validated
- string fields standardized
- data types explicitly defined

The Silver customer dataset is now ready to be used as an input for
downstream analysis and integration with the orders dataset.

# ORDERS – Silver Preprocessing

---

**Dataset:** `olist_orders_dataset.csv`  
**Silver output:** `silver_orders.csv`

This section uses `bronze_customers`, which was already loaded in the shared setup.

In [47]:

print("Shape:", bronze_orders.shape)
print("\nColumns:")
print(bronze_orders.columns.tolist())

print("\nData types:")
print(bronze_orders.dtypes)

print("\nFirst rows:")
display(bronze_orders.head())

print("\nMissing values:")
display(
    bronze_orders.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
    .assign(
        missing_pct=lambda x: (x["missing_count"] / len(bronze_orders) * 100).round(2)
    )
)

print("\nDuplicate rows:", bronze_orders.duplicated().sum())

Shape: (99441, 8)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

Data types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

First rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Missing values:


,missing_count,missing_pct
order_delivered_customer_date,2965,2.98
order_delivered_carrier_date,1783,1.79
order_approved_at,160,0.16
order_id,0,0.00
order_purchase_timestamp,0,0.00
order_status,0,0.00
customer_id,0,0.00
order_estimated_delivery_date,0,0.00



Duplicate rows: 0


## 1. Data Quality & Key Validation

Validate primary key uniqueness, customer references and order status values.

### Key Integrity

Validate the uniqueness of the order primary key and identify duplicate order records.

#### Key Definition

The `orders` dataset uses the following key structure:

- `id_orders`: technical surrogate key of the Silver layer, generated sequentially starting at 1.
- `order_id`: business/natural key identifying the original Olist order.
- `customer_id`: foreign key referencing `customer_id` in the customers dataset.

The surrogate key is introduced in the Silver layer, while the original `order_id` is preserved as the business key.

In [48]:
# Validate primary key uniqueness

print("Unique order_ids:", bronze_orders["order_id"].nunique())
print("Total rows:", len(bronze_orders))
print("Duplicate order_ids:", bronze_orders["order_id"].duplicated().sum())

customer_id_match = bronze_orders["customer_id"].isin(bronze_customers["customer_id"])

print("Orders with valid customer_id:", customer_id_match.sum())
print("Orders with missing customer reference:", (~customer_id_match).sum())


Unique order_ids: 99441
Total rows: 99441
Duplicate order_ids: 0
Orders with valid customer_id: 99441
Orders with missing customer reference: 0


### Order Status Distribution

Examine the distribution of order statuses to understand the composition of the order dataset and identify dominant order lifecycle stages.

In [49]:
print("Order status distribution:")
display(
    bronze_orders["order_status"]
    .value_counts()
    .to_frame("order_count")
    .assign(percentage=lambda x: (x["order_count"] / len(bronze_orders) * 100).round(2)))

Order status distribution:


,order_count,percentage
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


### Temporal Consistency

Validate the logical sequence of order events and identify records with impossible timestamp relationships.

##### Temporal Fields Preparation

Convert order lifecycle timestamps to datetime format before validating their temporal consistency.

In [50]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"]

bronze_orders[date_columns] = bronze_orders[date_columns].apply(pd.to_datetime)

bronze_orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [51]:
print("Approved before purchase:",(bronze_orders["order_approved_at"] < bronze_orders["order_purchase_timestamp"]).sum())

print("Carrier before purchase:",(bronze_orders["order_delivered_carrier_date"] < bronze_orders["order_purchase_timestamp"]).sum())

print("Customer delivery before purchase:",(bronze_orders["order_delivered_customer_date"] < bronze_orders["order_purchase_timestamp"]).sum())

print("Customer delivery before carrier:",(bronze_orders["order_delivered_customer_date"]< bronze_orders["order_delivered_carrier_date"]).sum())

Approved before purchase: 0
Carrier before purchase: 166
Customer delivery before purchase: 0
Customer delivery before carrier: 23


In [52]:
# Missing timestamp rates by order status

missing_pct_by_status = (bronze_orders.groupby("order_status")[date_columns].apply(lambda x: x.isna().mean() * 100).round(2))

display(missing_pct_by_status)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0.0,0.00,100.0,100.00,0.0
canceled,0.0,22.56,88.0,99.04,0.0
created,0.0,100.00,100.0,100.00,0.0
delivered,0.0,0.01,0.0,0.01,0.0
invoiced,0.0,0.00,100.0,100.00,0.0
processing,0.0,0.00,100.0,100.00,0.0
shipped,0.0,0.00,0.0,100.00,0.0
unavailable,0.0,0.00,100.0,100.00,0.0


### Temporal Anomaly Assessment

Quantify the magnitude of temporal inconsistencies to distinguish isolated data-quality issues from potentially systematic problems.

In [53]:
# Assess magnitude of temporal anomalies

# Carrier date before purchase
carrier_anomalies = bronze_orders.loc[
    bronze_orders["order_delivered_carrier_date"]
    < bronze_orders["order_purchase_timestamp"]
].copy()

carrier_anomalies["difference_minutes"] = (
    carrier_anomalies["order_purchase_timestamp"]
    - carrier_anomalies["order_delivered_carrier_date"]
).dt.total_seconds() / 60


# Customer delivery before carrier
delivery_anomalies = bronze_orders.loc[
    bronze_orders["order_delivered_customer_date"]
    < bronze_orders["order_delivered_carrier_date"]
].copy()

delivery_anomalies["difference_minutes"] = (
    delivery_anomalies["order_delivered_carrier_date"]
    - delivery_anomalies["order_delivered_customer_date"]
).dt.total_seconds() / 60


# Summarize anomaly magnitude
print("Carrier before purchase:")
display(
    carrier_anomalies["difference_minutes"].describe()
)

print("\nDelivery before carrier:")
display(
    delivery_anomalies["difference_minutes"].describe()
)

Carrier before purchase:


count       166.000000
mean       1562.368675
std       19135.430706
min           0.400000
25%          14.066667
50%          35.991667
75%          61.050000
max      246545.883333
Name: difference_minutes, dtype: float64


Delivery before carrier:


count       23.000000
mean      4707.307971
std       5358.484182
min         23.300000
25%       1400.300000
50%       2391.866667
75%       7725.541667
max      23178.483333
Name: difference_minutes, dtype: float64

In [54]:
# Review most extreme temporal anomalies

print("Carrier before purchase - largest deviations:")
display(
    carrier_anomalies[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "difference_minutes"
        ]
    ]
    .sort_values("difference_minutes", ascending=False)
    .head(5)
)

print("\nDelivery before carrier - largest deviations:")
display(
    delivery_anomalies[
        [
            "order_id",
            "order_status",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "difference_minutes"
        ]
    ]
    .sort_values("difference_minutes", ascending=False)
    .head(5)
)

Carrier before purchase - largest deviations:


,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date,difference_minutes
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-01-26 13:35:00,246545.883333
83321,4021cd7611d6d9ce5ffcd24817fc374f,delivered,2018-08-18 11:49:40,2018-08-14 06:22:00,6087.666667
67844,db090a16182b263b1e896bb26c6f66cf,delivered,2018-07-13 16:14:08,2018-07-13 13:59:00,135.133333
65452,9711d975b961355b4b5d636857e48498,delivered,2018-06-13 15:23:50,2018-06-13 13:15:00,128.833333
79401,89d32b64af005178b318f76cd60f2c3c,delivered,2018-07-06 11:54:40,2018-07-06 09:48:00,126.666667



Delivery before carrier - largest deviations:


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,difference_minutes
34939,c1e2bf2b7dd3309f2f5356c6b63968fa,delivered,2017-03-02 17:34:26,2017-02-14 15:15:57,23178.483333
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01,11345.700000
45302,29941903985f944b0ffc49c479c1547d,delivered,2017-06-09 15:07:29,2017-06-02 11:09:16,10318.216667
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10,8654.333333
49933,76458889992169d3135b264dc13aec67,delivered,2016-10-26 11:43:06,2016-10-20 18:03:17,8259.816667


### Data Quality Decision

Temporal inconsistencies affect a small number of records. Since the correct timestamps cannot be reliably inferred, the original values are retained.

Affected records are flagged and excluded from derived duration calculations where the affected timestamp is required.

### Missing Value Assessment & Treatment

Missing timestamps are largely consistent with the respective order status. For example, delivery-related timestamps are naturally absent for orders that have not reached the delivery stage.

A small number of delivered orders also contain missing timestamps. Since the correct event times cannot be reliably inferred, missing values are retained as `NaN` and no imputation or row deletion is performed.

In [55]:
# Missing values by order status and date column

missing_pct_by_status = (
    bronze_orders
    .groupby("order_status")[date_columns]
    .apply(lambda x: x.isna().mean() * 100)
    .round(2))

display(missing_pct_by_status)

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0.0,0.00,100.0,100.00,0.0
canceled,0.0,22.56,88.0,99.04,0.0
created,0.0,100.00,100.0,100.00,0.0
delivered,0.0,0.01,0.0,0.01,0.0
invoiced,0.0,0.00,100.0,100.00,0.0
processing,0.0,0.00,100.0,100.00,0.0
shipped,0.0,0.00,0.0,100.00,0.0
unavailable,0.0,0.00,100.0,100.00,0.0


In [56]:
# Investigate missing timestamps in delivered orders

delivered_missing_dates = bronze_orders[
    (bronze_orders["order_status"] == "delivered")
    & (
        bronze_orders[date_columns]
        .isna()
        .any(axis=1)
    )
].copy()

print("Delivered orders with missing timestamps:",len(delivered_missing_dates))

display(delivered_missing_dates[["order_id", "order_status"] + date_columns].head(5))

Delivered orders with missing timestamps: 23


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16


### Data Quality Summary

The order data is suitable for further analysis after applying explicit quality flags. Temporal inconsistencies are retained in the original data but excluded from affected duration calculations. Missing timestamps are retained as `NaN` where no reliable imputation is possible.

The resulting dataset preserves the original observations while preventing identified data-quality issues from distorting derived analytical features.

# 2. Temporal Quality Flags

Create flags for temporal inconsistencies without modifying the original timestamp columns.

### Temporal anomaly flags

In [57]:
# Create temporal quality flags

bronze_orders["carrier_before_purchase_flag"] = (bronze_orders["order_delivered_carrier_date"]<bronze_orders["order_purchase_timestamp"])

bronze_orders["delivery_before_carrier_flag"] = (bronze_orders["order_delivered_customer_date"]<bronze_orders["order_delivered_carrier_date"])

### Missing timestamp flags

In [58]:
# Create missing timestamp flags

bronze_orders["missing_approval_flag"] = (bronze_orders["order_approved_at"].isna())

bronze_orders["missing_carrier_date_flag"] = (bronze_orders["order_delivered_carrier_date"].isna())

bronze_orders["missing_delivery_date_flag"] = (bronze_orders["order_delivered_customer_date"].isna())

### Flag validation

In [59]:
# Validate temporal quality flags

print("Carrier before purchase:",bronze_orders["carrier_before_purchase_flag"].sum())

print("Delivery before carrier:",bronze_orders["delivery_before_carrier_flag"].sum())

print("Missing approval timestamp:",bronze_orders["missing_approval_flag"].sum())

print("Missing carrier timestamp:",bronze_orders["missing_carrier_date_flag"].sum())

print("Missing customer delivery timestamp:",bronze_orders["missing_delivery_date_flag"].sum())

Carrier before purchase: 166
Delivery before carrier: 23
Missing approval timestamp: 160
Missing carrier timestamp: 1783
Missing customer delivery timestamp: 2965


### Quality flag summary

In [60]:
# Summary of quality flags

quality_flag_summary = pd.DataFrame({
    "flag": [
        "carrier_before_purchase_flag",
        "delivery_before_carrier_flag",
        "missing_approval_flag",
        "missing_carrier_date_flag",
        "missing_delivery_date_flag"],
    "count": [
        bronze_orders["carrier_before_purchase_flag"].sum(),
        bronze_orders["delivery_before_carrier_flag"].sum(),
        bronze_orders["missing_approval_flag"].sum(),
        bronze_orders["missing_carrier_date_flag"].sum(),
        bronze_orders["missing_delivery_date_flag"].sum()]})

quality_flag_summary["percentage"] = (quality_flag_summary["count"] / len(bronze_orders) * 100).round(3)

display(quality_flag_summary)

,flag,count,percentage
0,carrier_before_purchase_flag,166,0.167
1,delivery_before_carrier_flag,23,0.023
2,missing_approval_flag,160,0.161
3,missing_carrier_date_flag,1783,1.793
4,missing_delivery_date_flag,2965,2.982


# 3. Silver Data Validation Orders

Validate the cleaned orders dataset before creating the Silver layer.

The Silver layer preserves the original order-level data and includes data-quality flags identified during preprocessing. Business features are intentionally excluded and will be created later in the Gold layer.

In [61]:
# Create Silver dataset from validated orders data
silver_orders = bronze_orders.copy()

# Create silver unique key
silver_orders.insert(
    0,
    "id_orders",
    range(1, len(silver_orders) + 1)
)

# Expected Silver schema
expected_silver_columns = [
    "id_orders",
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "carrier_before_purchase_flag",
    "delivery_before_carrier_flag",
    "missing_approval_flag",
    "missing_carrier_date_flag",
    "missing_delivery_date_flag"
]


# Validate row count against raw dataset
assert len(silver_orders) == orders_raw_row_count

# Validate silver unique key
assert silver_orders["id_orders"].notna().all()
assert silver_orders["id_orders"].is_unique
assert silver_orders["id_orders"].min() == 1
assert silver_orders["id_orders"].max() == len(silver_orders)


# Validate primary key
assert silver_orders["order_id"].is_unique


# Reorder columns according to the Silver schema
silver_orders = silver_orders[expected_silver_columns]


# Validate expected columns
assert list(silver_orders.columns) == expected_silver_columns


# Validate required key fields
assert silver_orders["order_id"].notna().all()
assert silver_orders["customer_id"].notna().all()
assert silver_orders["order_status"].notna().all()
assert silver_orders["order_purchase_timestamp"].notna().all()


# Validate temporal data types
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

assert all(
    pd.api.types.is_datetime64_any_dtype(silver_orders[column])
    for column in date_columns
)


# Validate quality flag data types
quality_flags = [
    "carrier_before_purchase_flag",
    "delivery_before_carrier_flag",
    "missing_approval_flag",
    "missing_carrier_date_flag",
    "missing_delivery_date_flag"
]

assert all(
    silver_orders[column].dtype == bool
    for column in quality_flags
)


# Validate temporal anomaly flags
assert (
    (
        silver_orders["carrier_before_purchase_flag"]
        != (
            silver_orders["order_delivered_carrier_date"]
            < silver_orders["order_purchase_timestamp"]
        )
    ).sum()
    == 0
)

assert (
    (
        silver_orders["delivery_before_carrier_flag"]
        != (
            silver_orders["order_delivered_customer_date"]
            < silver_orders["order_delivered_carrier_date"]
        )
    ).sum()
    == 0
)


# Validate missing timestamp flags
assert (
    (
        silver_orders["missing_approval_flag"]
        != silver_orders["order_approved_at"].isna()
    ).sum()
    == 0
)

assert (
    (
        silver_orders["missing_carrier_date_flag"]
        != silver_orders["order_delivered_carrier_date"].isna()
    ).sum()
    == 0
)

assert (
    (
        silver_orders["missing_delivery_date_flag"]
        != silver_orders["order_delivered_customer_date"].isna()
    ).sum()
    == 0
)


# Create Silver quality summary
silver_quality_summary = (
    silver_orders[quality_flags]
    .sum()
    .to_frame("flagged_count")
)

silver_quality_summary["flagged_pct"] = (
    silver_quality_summary["flagged_count"]
    / len(silver_orders)
    * 100
).round(3)


# Final Silver validation
assert len(silver_orders) == orders_raw_row_count
assert silver_orders["order_id"].is_unique
assert list(silver_orders.columns) == expected_silver_columns


print("Silver data validation passed.")
print(f"Rows: {len(silver_orders):,}")
print(f"Unique order_ids: {silver_orders['order_id'].nunique():,}")
print(f"Columns: {len(silver_orders.columns)}")
print("Silver schema column order:")
print(list(silver_orders.columns))

Silver data validation passed.
Rows: 99,441
Unique order_ids: 99,441
Columns: 14
Silver schema column order:
['id_orders', 'order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'carrier_before_purchase_flag', 'delivery_before_carrier_flag', 'missing_approval_flag', 'missing_carrier_date_flag', 'missing_delivery_date_flag']


# 4. Orders – Preprocessing Summary

The orders dataset was validated and prepared for the Silver layer.

### Data Layers

The preprocessing follows a simple Bronze-to-Silver architecture.

- **Bronze:** `bronze_orders` and `bronze_customers` contain the loaded source data.
- **Silver:** `silver_orders` contains the validated order-level data, including technical and data-quality fields.
- **Gold:** Business features and analytical KPIs will be created in a separate Gold layer.

### Data quality

- The `order_id` primary key is unique across all records.
- Customer references were validated against the customer dataset.
- Missing timestamps were assessed in the context of order status.
- Temporal inconsistencies were identified and flagged rather than silently corrected.
- Original timestamp values were preserved.

### Quality flags

The following data-quality flags were created:

- `carrier_before_purchase_flag`
- `delivery_before_carrier_flag`
- `missing_approval_flag`
- `missing_carrier_date_flag`
- `missing_delivery_date_flag`

Temporal inconsistencies remain in the original data and are explicitly identified through quality flags.

Missing timestamp values were retained as `NaN` where no reliable imputation was possible.

### Silver layer

The resulting Silver dataset preserves the original order-level observations and adds explicit data-quality indicators.

Business features and analytical KPIs are intentionally not included in the Silver layer. These will be created in the separate Gold layer based on the validated Silver data.

The Silver dataset is ready for downstream data modeling, feature engineering and business analysis.

# ORDER REVIEWS – Silver Preprocessing

---

**Dataset:** `olist_order_reviews_dataset.csv`  
**Silver output:** `silver_order_reviews_en.csv`

This section remains functionally separated from Customers and Orders.

## 1. Data Understanding

Inspect the structure, granularity, data types, and basic characteristics of the order reviews dataset before applying any transformations.

The objective is to understand how reviews are represented in the raw data and identify potential data quality issues that require further investigation.

### Data Overview

In [62]:
bronze_order_reviews.shape

(99224, 9)

In [63]:
bronze_order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,NaN,Recebi bem antes do prazo estipulado.,I received it well before the promised deadline.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,NaN,Parabéns lojas lannister adorei comprar pela I...,"Congratulations, Lannister stores! I loved buy...",2018-03-01 00:00:00,2018-03-02 10:26:53


In [64]:
bronze_order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   review_id                  99224 non-null  str  
 1   order_id                   99224 non-null  str  
 2   review_score               99224 non-null  int64
 3   review_comment_title       11568 non-null  str  
 4   review_comment_title_en    11564 non-null  str  
 5   review_comment_message     40977 non-null  str  
 6   review_comment_message_en  40947 non-null  str  
 7   review_creation_date       99224 non-null  str  
 8   review_answer_timestamp    99224 non-null  str  
dtypes: int64(1), str(8)
memory usage: 22.5 MB


In [65]:
bronze_order_reviews.describe(include="all")

,review_id,order_id,review_score,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en,review_creation_date,review_answer_timestamp
count,99224,99224,99224.000000,11568,11564,40977,40947,99224,99224
unique,98410,98673,NaN,4527,3219,36159,34189,636,98248
top,c444278834184f72b1484dfe47de7f97,c88b1d1b157a9999ce368f218a407141,NaN,Recomendo,Great,Muito bom,Very good,2017-12-19 00:00:00,2017-06-15 23:21:05
freq,3,3,NaN,423,963,230,391,463,4
mean,NaN,NaN,4.086421,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,1.347579,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,4.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN,NaN,NaN


### Schema & Column Overview

In [66]:
bronze_order_reviews.columns.tolist()

['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_title_en',
 'review_comment_message',
 'review_comment_message_en',
 'review_creation_date',
 'review_answer_timestamp']

In [67]:

expected_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en",
    "review_creation_date",
    "review_answer_timestamp"
]

assert bronze_order_reviews.columns.tolist() == expected_columns

In [68]:
bronze_order_reviews.dtypes

review_id                      str
order_id                       str
review_score                 int64
review_comment_title           str
review_comment_title_en        str
review_comment_message         str
review_comment_message_en      str
review_creation_date           str
review_answer_timestamp        str
dtype: object

### Granularity & Relationships

In [69]:
#review_id
print("Unique review_id:", bronze_order_reviews["review_id"].nunique())
print("Rows with duplicated review_id:", bronze_order_reviews["review_id"].duplicated().sum())

Unique review_id: 98410
Rows with duplicated review_id: 814


In [70]:
#order_id
print("Unique order_id:", bronze_order_reviews["order_id"].nunique())
print("Rows with duplicated order_id:", bronze_order_reviews["order_id"].duplicated().sum())

Unique order_id: 98673
Rows with duplicated order_id: 551


In [71]:
review_order_counts = (
    bronze_order_reviews
    .groupby("review_id")["order_id"]
    .nunique()
    .value_counts()
    .sort_index()
)

review_order_counts

order_id
1    97621
2      764
3       25
Name: count, dtype: int64

In [72]:
order_review_counts = (
    bronze_order_reviews
    .groupby("order_id")["review_id"]
    .nunique()
    .value_counts()
    .sort_index()
)

order_review_counts

review_id
1    98126
2      543
3        4
Name: count, dtype: int64

### Review Score Distribution

In [73]:
bronze_order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [74]:
bronze_order_reviews["review_score"].describe()

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

### Date & Time Characteristics

In [75]:
bronze_order_reviews[
    ["review_creation_date", "review_answer_timestamp"]
].dtypes

review_creation_date       str
review_answer_timestamp    str
dtype: object

In [76]:
bronze_order_reviews[
    ["review_creation_date", "review_answer_timestamp"]
].head()

,review_creation_date,review_answer_timestamp
0,2018-01-18 00:00:00,2018-01-18 21:46:59
1,2018-03-10 00:00:00,2018-03-11 03:05:13
2,2018-02-17 00:00:00,2018-02-18 14:36:24
3,2017-04-21 00:00:00,2017-04-21 22:02:06
4,2018-03-01 00:00:00,2018-03-02 10:26:53


In [77]:
review_creation_dt = pd.to_datetime(
    bronze_order_reviews["review_creation_date"],
    errors="coerce"
)

review_answer_dt = pd.to_datetime(
    bronze_order_reviews["review_answer_timestamp"],
    errors="coerce"
)

In [78]:
print(
    "Review creation:",
    review_creation_dt.min(),
    "to",
    review_creation_dt.max()
)

print(
    "Review answer:",
    review_answer_dt.min(),
    "to",
    review_answer_dt.max()
)

Review creation: 2016-10-02 00:00:00 to 2018-08-31 00:00:00
Review answer: 2016-10-07 18:32:28 to 2018-10-29 12:27:35


### Chapter Summary

The order reviews dataset contains 99,224 records across seven columns covering review identifiers, order identifiers, review scores, review text, and review timestamps.

The analysis shows that review IDs and order IDs are not strictly unique. Most review IDs are associated with one order, while a smaller number are associated with multiple orders. Conversely, most orders contain one review record, while a limited number contain multiple review records. Therefore, neither identifier is sufficient to define a unique row across the complete dataset.

Review scores range from 1 to 5, with higher scores occurring substantially more frequently than lower scores.

The review creation dates cover the period from October 2016 to August 2018, while review answer timestamps extend from October 2016 to October 2018.

The raw dataset stores the timestamp fields as strings. Their values follow a consistent datetime representation and will be evaluated further during the data quality assessment and transformation phase.

Overall, the dataset structure and granularity are sufficiently understood to proceed to a dedicated data quality assessment before applying any transformations.

# 2. Data Quality Assessment

Evaluate the data quality of the order reviews dataset before applying transformations.

The assessment focuses on missing values, duplicate records, valid value ranges, datetime consistency, and text field quality. The objective is to identify issues that require treatment during preprocessing while preserving valid characteristics of the raw dataset.

### Missing Value Assessment

In [79]:
missing_summary = (
    bronze_order_reviews.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"]
    / len(bronze_order_reviews)
    * 100
)

missing_summary

,missing_count,missing_pct
review_id,0,0.000000
order_id,0,0.000000
review_score,0,0.000000
review_comment_title,87656,88.341530
review_comment_title_en,87660,88.345562
review_comment_message,58247,58.702532
review_comment_message_en,58277,58.732766
review_creation_date,0,0.000000
review_answer_timestamp,0,0.000000


In [80]:
mandatory_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
]

bronze_order_reviews[mandatory_columns].isna().sum()

review_id                  0
order_id                   0
review_score               0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [81]:
assert bronze_order_reviews[mandatory_columns].notna().all().all()

In [82]:
comment_columns = [
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en"
]

bronze_order_reviews[comment_columns].isna().sum()

review_comment_title         87656
review_comment_title_en      87660
review_comment_message       58247
review_comment_message_en    58277
dtype: int64

In [83]:
empty_text_summary = {}

for column in comment_columns:
    empty_text_summary[column] = {
        "empty_string": (bronze_order_reviews[column] == "").sum(),
        "whitespace_only":bronze_order_reviews[column].str.strip().eq("").sum()
    }

empty_text_summary

{'review_comment_title': {'empty_string': np.int64(0),
  'whitespace_only': np.int64(2)},
 'review_comment_title_en': {'empty_string': np.int64(0),
  'whitespace_only': np.int64(0)},
 'review_comment_message': {'empty_string': np.int64(0),
  'whitespace_only': np.int64(27)},
 'review_comment_message_en': {'empty_string': np.int64(0),
  'whitespace_only': np.int64(0)}}

### Duplicate Record Validation

In [84]:
exact_duplicates = bronze_order_reviews.duplicated().sum()

exact_duplicates

np.int64(0)

In [85]:
review_id_duplicate_rows = bronze_order_reviews["review_id"].duplicated().sum()

review_id_duplicate_rows

np.int64(814)

In [86]:
order_id_duplicate_rows = bronze_order_reviews["order_id"].duplicated().sum()

order_id_duplicate_rows

np.int64(551)

### Value Validation

In [87]:
bronze_order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [88]:
assert bronze_order_reviews["review_score"].between(1, 5).all() #kapitel3

### Datetime validation

In [89]:
review_creation_dt = pd.to_datetime(
    bronze_order_reviews["review_creation_date"],
    errors="coerce"
)

review_answer_dt = pd.to_datetime(
    bronze_order_reviews["review_answer_timestamp"],
    errors="coerce"
)

print("Invalid review_creation_date:", review_creation_dt.isna().sum())
print("Invalid review_answer_timestamp:", review_answer_dt.isna().sum())

Invalid review_creation_date: 0
Invalid review_answer_timestamp: 0


In [90]:
assert review_creation_dt.notna().all()
assert review_answer_dt.notna().all()

In [91]:
answer_before_creation = review_answer_dt < review_creation_dt

answer_before_creation.sum()

np.int64(0)

In [92]:
assert not answer_before_creation.any()

### Text Field Validation

In [93]:
blank_text_summary = {}

for column in comment_columns:
    blank_text_summary[column] = (
        bronze_order_reviews[column].notna()
        & bronze_order_reviews[column].str.strip().eq("")
    ).sum()

blank_text_summary

{'review_comment_title': np.int64(2),
 'review_comment_title_en': np.int64(0),
 'review_comment_message': np.int64(27),
 'review_comment_message_en': np.int64(0)}

### Chapter Summary

The data quality assessment identified no missing values in mandatory identifiers, review scores, or timestamp fields.

The review comment fields contain substantial missingness, which is treated as expected because review text is optional. A small number of non-null comment values contain only whitespace and will be normalized during preprocessing.

No exact duplicate rows were identified. Repeated `review_id` and `order_id` values do not represent exact duplicate records and therefore do not require record removal.

All review scores are within the valid range of 1 to 5.

Both timestamp fields can be successfully parsed as datetime values, and no chronological inconsistencies were identified between review creation and answer timestamps.

Overall, no critical data quality issues requiring record-level removal were identified. The main preprocessing requirements are the normalization of text fields and conversion of timestamp fields to appropriate datetime types.

# 3. Data Transformation

Apply the required transformations identified during the data quality assessment to create a clean Silver version of the order reviews dataset.

The transformations focus on text normalization and datetime conversion while preserving the original review content and row-level structure of the dataset.

### Create Silver Dataset

In [94]:
silver_order_reviews = bronze_order_reviews.copy()

silver_order_reviews.shape

(99224, 9)

In [95]:
silver_order_reviews["id_order_reviews"] = range(
    1,
    len(silver_order_reviews) + 1
)

silver_order_reviews.insert(
    0,
    "id_order_reviews",
    silver_order_reviews.pop("id_order_reviews")
)

In [96]:
assert silver_order_reviews["id_order_reviews"].is_unique
assert silver_order_reviews["id_order_reviews"].min() == 1
assert silver_order_reviews["id_order_reviews"].max() == len(silver_order_reviews)

### Text Normalization

In [97]:
comment_columns = [
    "review_comment_title",
    "review_comment_message"
]

for column in comment_columns:
    silver_order_reviews[column] = (
        silver_order_reviews[column]
        .astype("string")
        .str.lower()
        .str.replace(r"[^a-z\s.,!?]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

In [98]:
translation_columns = [
    "review_comment_title_en",
    "review_comment_message_en"
]

for column in translation_columns:
    silver_order_reviews[column] = (
        silver_order_reviews[column]
        .astype("string")
        .str.lower()
        .str.replace(r"[^a-z\s.,!?]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

In [99]:
silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nota dez",
    "review_comment_title_en"
] = "ranking ten"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nota",
    "review_comment_title_en"
] = "ranking"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nota",
    "review_comment_message_en"
] = "ranking"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nenhuma",
    "review_comment_title_en"
] = "none"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nenhuma",
    "review_comment_message_en"
] = "none"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nd",
    "review_comment_message_en"
] = "not available"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "no h",
    "review_comment_message_en"
] = "there is no"

In [100]:
for column in comment_columns + translation_columns:
    non_null_values = silver_order_reviews[column].dropna()

    assert (
        non_null_values
        == non_null_values.str.strip()
    ).all()

    assert (
        non_null_values
        == non_null_values.str.lower()
    ).all()

    assert (
        non_null_values
        .str.match(r"^[a-z\s.,!?]+$")
        .all()
    )

In [101]:
assert (
    silver_order_reviews["review_comment_title"].isna()
    == silver_order_reviews["review_comment_title_en"].isna()
).all()

assert (
    silver_order_reviews["review_comment_message"].isna()
    == silver_order_reviews["review_comment_message_en"].isna()
).all()

In [102]:
silver_order_reviews[
    [
        "review_comment_title",
        "review_comment_title_en",
        "review_comment_message",
        "review_comment_message_en"
    ]
].head(10)

,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,recebi bem antes do prazo estipulado.,i received it well before the promised deadline.
4,<NA>,<NA>,parabns lojas lannister adorei comprar pela in...,"congratulations, lannister stores! i loved buy..."
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,recomendo,i recommend,aparelho eficiente. no site a marca do aparelh...,"efficient device. on the site, the devices bra..."


### Datetime Transformation

In [103]:
datetime_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in datetime_columns:
    silver_order_reviews[column] = pd.to_datetime(
        silver_order_reviews[column],
        errors="raise"
    )

In [104]:
silver_order_reviews[datetime_columns].dtypes

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [105]:
assert silver_order_reviews[datetime_columns].notna().all().all()

In [106]:
answer_before_creation = (
    silver_order_reviews["review_answer_timestamp"]
    < silver_order_reviews["review_creation_date"]
)

assert not answer_before_creation.any()

### Transformation Validation

In [107]:
silver_order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id_order_reviews           99224 non-null  int64         
 1   review_id                  99224 non-null  str           
 2   order_id                   99224 non-null  str           
 3   review_score               99224 non-null  int64         
 4   review_comment_title       11175 non-null  string        
 5   review_comment_title_en    11175 non-null  string        
 6   review_comment_message     40886 non-null  string        
 7   review_comment_message_en  40886 non-null  string        
 8   review_creation_date       99224 non-null  datetime64[us]
 9   review_answer_timestamp    99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(2), str(2), string(4)
memory usage: 19.4 MB


In [108]:
assert len(silver_order_reviews) == len(bronze_order_reviews)
assert silver_order_reviews["review_id"].nunique() == bronze_order_reviews["review_id"].nunique()
assert silver_order_reviews["order_id"].nunique() == bronze_order_reviews["order_id"].nunique()

### Silver Dataset Overview

In [109]:
silver_order_reviews.head()

,id_order_reviews,review_id,order_id,review_score,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en,review_creation_date,review_answer_timestamp
0,1,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,<NA>,<NA>,<NA>,<NA>,2018-01-18,2018-01-18 21:46:59
1,2,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,<NA>,<NA>,<NA>,<NA>,2018-03-10,2018-03-11 03:05:13
2,3,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,<NA>,<NA>,<NA>,<NA>,2018-02-17,2018-02-18 14:36:24
3,4,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,<NA>,<NA>,recebi bem antes do prazo estipulado.,i received it well before the promised deadline.,2017-04-21,2017-04-21 22:02:06
4,5,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,<NA>,<NA>,parabns lojas lannister adorei comprar pela in...,"congratulations, lannister stores! i loved buy...",2018-03-01,2018-03-02 10:26:53


In [110]:
silver_order_reviews.dtypes

id_order_reviews                      int64
review_id                               str
order_id                                str
review_score                          int64
review_comment_title                 string
review_comment_title_en              string
review_comment_message               string
review_comment_message_en            string
review_creation_date         datetime64[us]
review_answer_timestamp      datetime64[us]
dtype: object

#  Order Reviews Dataset Summary

The order reviews dataset was transformed into a clean Silver version while preserving the original row-level structure and review identifiers.

The Silver dataset receives a unique key `id_order_reviews` to uniquely identify each review record. The key starts at 1 and preserves the row-level grain of the dataset. The existing `review_id` and `order_id` fields are retained as business identifiers and relationship keys.

The Portuguese review text fields were normalized by converting text to lowercase, removing leading and trailing whitespace, standardizing whitespace, and removing characters outside the defined character set. Empty text values were converted to missing values. The original Portuguese review content was retained.

The English review title and message fields were normalized using the same text transformation rules. Missing English translations for specific rating-related values were completed using defined mappings, while the Portuguese original values remained unchanged. This ensured consistency between the original and translated review fields.

The review creation date and answer timestamp were converted from strings to datetime values. The resulting timestamps remain complete and chronologically consistent.

No records were removed and no identifier values were altered during the transformation.

The resulting Silver dataset preserves the original review information while providing standardized Portuguese and English text fields and appropriately typed timestamp columns. It is now prepared for downstream analysis and integration with the other Olist datasets.

# 5. Final Silver Validation & Common Export

All three Silver datasets are finally validated after their respective transformations.

The export is performed **only once at the end** for all three datasets.

In [111]:
# ============================================================
# Final validation – Customers
# ============================================================

expected_customers_columns = [
    "id_customers",
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
]

assert silver_customers.columns.tolist() == expected_customers_columns
assert len(silver_customers) == customers_raw_row_count

assert silver_customers["id_customers"].notna().all()
assert silver_customers["id_customers"].is_unique
assert silver_customers["id_customers"].min() == 1
assert silver_customers["id_customers"].max() == len(silver_customers)

assert silver_customers["customer_id"].notna().all()
assert silver_customers["customer_id"].is_unique
assert silver_customers["customer_unique_id"].notna().all()
assert silver_customers["customer_city"].notna().all()
assert silver_customers["customer_state"].notna().all()
assert silver_customers.isna().sum().sum() == 0

assert invalid_states == set()

assert silver_customers["customer_zip_code_prefix"].str.len().eq(5).all()
assert silver_customers["customer_zip_code_prefix"].str.isnumeric().all()

print("Customers Silver validation passed.")


Customers Silver validation passed.


In [112]:
# ============================================================
# Final validation – Orders
# ============================================================

expected_orders_columns = [
    "id_orders",
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "carrier_before_purchase_flag",
    "delivery_before_carrier_flag",
    "missing_approval_flag",
    "missing_carrier_date_flag",
    "missing_delivery_date_flag"
]

assert silver_orders.columns.tolist() == expected_orders_columns
assert len(silver_orders) == orders_raw_row_count

assert silver_orders["id_orders"].notna().all()
assert silver_orders["id_orders"].is_unique
assert silver_orders["id_orders"].min() == 1
assert silver_orders["id_orders"].max() == len(silver_orders)

assert silver_orders["order_id"].notna().all()
assert silver_orders["order_id"].is_unique
assert silver_orders["customer_id"].notna().all()
assert silver_orders["order_status"].notna().all()
assert silver_orders["order_purchase_timestamp"].notna().all()

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

assert all(
    pd.api.types.is_datetime64_any_dtype(silver_orders[column])
    for column in date_columns
)

quality_flags = [
    "carrier_before_purchase_flag",
    "delivery_before_carrier_flag",
    "missing_approval_flag",
    "missing_carrier_date_flag",
    "missing_delivery_date_flag"
]

assert all(
    silver_orders[column].dtype == bool
    for column in quality_flags
)

print("Orders Silver validation passed.")


Orders Silver validation passed.


In [113]:
# ============================================================
# Final validation – Order Reviews
# ============================================================

expected_order_reviews_columns = [
    "id_order_reviews",
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en",
    "review_creation_date",
    "review_answer_timestamp"
]

assert silver_order_reviews.columns.tolist() == expected_order_reviews_columns
assert len(silver_order_reviews) == order_reviews_raw_row_count

assert silver_order_reviews["id_order_reviews"].notna().all()
assert silver_order_reviews["id_order_reviews"].is_unique
assert silver_order_reviews["id_order_reviews"].min() == 1
assert silver_order_reviews["id_order_reviews"].max() == len(silver_order_reviews)

mandatory_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
]

assert silver_order_reviews[mandatory_columns].notna().all().all()
assert silver_order_reviews["review_score"].between(1, 5).all()

assert (
    silver_order_reviews["review_comment_title"].isna()
    == silver_order_reviews["review_comment_title_en"].isna()
).all()

assert (
    silver_order_reviews["review_comment_message"].isna()
    == silver_order_reviews["review_comment_message_en"].isna()
).all()

assert pd.api.types.is_datetime64_any_dtype(
    silver_order_reviews["review_creation_date"]
)

assert pd.api.types.is_datetime64_any_dtype(
    silver_order_reviews["review_answer_timestamp"]
)

assert (
    silver_order_reviews["review_answer_timestamp"]
    >= silver_order_reviews["review_creation_date"]
).all()

print("Order Reviews Silver validation passed.")


Order Reviews Silver validation passed.


### 5.1 Common Silver Export


In [ ]:
# Export all validated Silver datasets once

# customers_output = PROCESSED_PATH / "silver_customers.csv"
# orders_output = PROCESSED_PATH / "silver_orders.csv"
# order_reviews_output = PROCESSED_PATH / "silver_order_reviews_en.csv"

# silver_customers.to_csv(
#    customers_output,
#    index=False
# )

# silver_orders.to_csv(
#    orders_output,
#    index=False
# )

# silver_order_reviews.to_csv(
#   order_reviews_output,
#   index=False
# )

# print("Silver datasets exported successfully:")
# print(f"- Customers:     {customers_output}")
# print(f"- Orders:        {orders_output}")
# print(f"- Order Reviews: {order_reviews_output}")



### S3 SILVER EXPORT – FUTURE AWS LAMBDA VERSION

This block is intentionally commented out.
It will be activated when the preprocessing pipeline is moved from local processing to AWS Lambda.

All three validated Silver datasets are uploaded to:

s3://dsi-final-project-360964564955-eu-central-1-an/processed/


In [ ]:

# import boto3
# from io import StringIO


# ------------------------------------------------------------
# S3 Configuration
# ------------------------------------------------------------

# S3_BUCKET = "dsi-final-project-360964564955-eu-central-1-an"
# PROCESSED_PREFIX = "processed/"


# ------------------------------------------------------------
# S3 Client
# ------------------------------------------------------------

# s3 = boto3.client("s3")


# ------------------------------------------------------------
# Helper function: Upload DataFrame to S3 as CSV
# ------------------------------------------------------------

# def save_csv_to_s3(df, bucket, key):
#
#     csv_buffer = StringIO()
#
#     df.to_csv(
#         csv_buffer,
#         index=False
#     )
#
#     s3.put_object(
#         Bucket=bucket,
#         Key=key,
#         Body=csv_buffer.getvalue(),
#         ContentType="text/csv"
#     )
#
#     print(f"Successfully uploaded: s3://{bucket}/{key}")


# ------------------------------------------------------------
# Export Silver datasets to S3
# ------------------------------------------------------------

# save_csv_to_s3(
#     silver_customers,
#     S3_BUCKET,
#     f"{PROCESSED_PREFIX}silver_customers.csv"
# )

# save_csv_to_s3(
#     silver_orders,
#     S3_BUCKET,
#     f"{PROCESSED_PREFIX}silver_orders.csv"
# )

# save_csv_to_s3(
#     silver_order_reviews,
#     S3_BUCKET,
#     f"{PROCESSED_PREFIX}silver_order_reviews_en.csv"
# )


# print("All Silver datasets successfully exported to S3.")

# 6. Combined Silver Layer Summary

This notebook combines the three existing preprocessing pipelines into one common execution:

- **Customers:** Standardization of IDs, City, State and ZIP Prefix, including `id_customers`.
- **Orders:** Validation of Order/Customer relationships, datetime conversion, temporal quality flags and `id_orders`.
- **Order Reviews:** Text normalization, defined translation adjustments, datetime conversion and `id_order_reviews`.

The three datasets remain clearly separated within the notebook. The shared setup and common export are executed only once.